# BDA Project — PersonaPath LDA Pipeline 
**Catalog:** `msbabigdata` | **Schema:** `default`  
**Output table:** `msbabigdata.default.lda_topic_results`

## Cell 1 — Load Yelp Dataset from Volume

In [0]:
import tarfile, os, shutil

tar_path   = "/Volumes/msbabigdata/trends/trendsmarket/yelp_dataset/yelp_dataset.tar"
temp_path  = "/tmp/yelp_dataset/"
final_path = "/Volumes/msbabigdata/trends/trendsmarket/yelp_dataset/extracted/"

os.makedirs(temp_path, exist_ok=True)

# Only extract if not already done
if not os.path.exists(final_path) or len(os.listdir(final_path)) == 0:
    os.makedirs(final_path, exist_ok=True)
    with tarfile.open(tar_path, 'r:*') as tar:
        tar.extractall(temp_path)
    print("Extracted:", os.listdir(temp_path))
    for file in os.listdir(temp_path):
        shutil.copy(os.path.join(temp_path, file), os.path.join(final_path, file))
    print("Copied to Volume:", os.listdir(final_path))
else:
    print("Already extracted:", os.listdir(final_path))

Already extracted: ['Dataset_User_Agreement.pdf', 'yelp_academic_dataset_business.json', 'yelp_academic_dataset_checkin.json', 'yelp_academic_dataset_review.json', 'yelp_academic_dataset_tip.json', 'yelp_academic_dataset_user.json']


## Cell 2 — Read JSON Files into Spark

In [0]:
BASE_PATH = "/Volumes/msbabigdata/trends/trendsmarket/yelp_dataset/extracted/"

business_df = spark.read.json(BASE_PATH + "yelp_academic_dataset_business.json")
review_df   = spark.read.json(BASE_PATH + "yelp_academic_dataset_review.json")
user_df     = spark.read.json(BASE_PATH + "yelp_academic_dataset_user.json")

print("business:", business_df.count())
print("reviews :", review_df.count())
print("users   :", user_df.count())

business: 150346
reviews : 6990280
users   : 1987897


## Cell 3 — Register Temp Views

In [0]:
business_df.createOrReplaceTempView("businesses")
review_df.createOrReplaceTempView("reviews")
user_df.createOrReplaceTempView("users")
print("Views registered: businesses, reviews, users")

Views registered: businesses, reviews, users


## Cell 4 — EDA: Top Cities

In [0]:
%sql
SELECT city, state,
    COUNT(*)            AS business_count,
    SUM(review_count)   AS total_reviews,
    ROUND(AVG(stars),2) AS avg_stars
FROM businesses
GROUP BY city, state
ORDER BY business_count DESC
LIMIT 20;

city,state,business_count,total_reviews,avg_stars
Philadelphia,PA,14567,936207,3.62
Tucson,AZ,9249,387239,3.59
Tampa,FL,9048,439450,3.58
Indianapolis,IN,7540,349228,3.58
Nashville,TN,6968,440988,3.64
New Orleans,LA,6208,621330,3.82
Reno,NV,5932,334555,3.76
Edmonton,AB,5054,98204,3.44
Saint Louis,MO,4827,244360,3.59
Santa Barbara,CA,3829,262853,4.05


## Cell 5 — EDA: Review Count Buckets (Philadelphia)

In [0]:
%sql
SELECT
    CASE
        WHEN review_count < 10  THEN '1_0-9'
        WHEN review_count < 25  THEN '2_10-24'
        WHEN review_count < 50  THEN '3_25-49'
        WHEN review_count < 100 THEN '4_50-99'
        WHEN review_count < 250 THEN '5_100-249'
        WHEN review_count < 500 THEN '6_250-499'
        ELSE                         '7_500+'
    END                             AS review_bucket,
    COUNT(*)                        AS business_count,
    ROUND(COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER(), 2)  AS pct_of_total
FROM businesses
WHERE city = 'Philadelphia'
GROUP BY review_bucket
ORDER BY review_bucket;

review_bucket,business_count,pct_of_total
1_0-9,4112,28.22
2_10-24,4215,28.93
3_25-49,2373,16.29
4_50-99,1732,11.89
5_100-249,1376,9.44
6_250-499,485,3.33
7_500+,276,1.89


## Cell 6 — EDA: Choose Review Count Threshold

In [0]:
%sql
SELECT threshold, COUNT(*) AS businesses_remaining
FROM businesses
CROSS JOIN (SELECT explode(array(10, 25, 50, 100, 200)) AS threshold)
WHERE city = 'Philadelphia'
  AND review_count >= threshold
GROUP BY threshold
ORDER BY threshold;

threshold,businesses_remaining
10,10457
25,6242
50,3869
100,2137
200,1039


## Cell 7 — Filter Philadelphia Businesses + Extract is_open & hours

Added `is_open` filter and `hours` extraction.  
- `is_open = 1` ensures recommender never suggests a closed business  
- `hours_*` fields flow into `business_profiles` so Layer 4 LLM can say "open until 11pm Fridays"


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW businesses_filtered AS
SELECT
    business_id,
    name,
    address,
    city,
    state,
    postal_code,
    stars,
    review_count,
    categories,
    is_open,
    hours.Monday    AS hours_monday,
    hours.Tuesday   AS hours_tuesday,
    hours.Wednesday AS hours_wednesday,
    hours.Thursday  AS hours_thursday,
    hours.Friday    AS hours_friday,
    hours.Saturday  AS hours_saturday,
    hours.Sunday    AS hours_sunday
FROM businesses
WHERE city        = 'Philadelphia'
  AND review_count >= 50
  AND is_open      = 1;

SELECT COUNT(*) AS open_philly_businesses FROM businesses_filtered;

open_philly_businesses
2800


## Cell 8 — Merge Reviews + Businesses + Users

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW merged AS
SELECT
    r.review_id,
    r.date          AS review_date,
    r.stars         AS review_stars,
    r.text          AS review_text,
    r.useful, r.funny, r.cool,
    b.business_id,
    b.name          AS business_name,
    b.address, b.city, b.state, b.postal_code,
    b.stars         AS business_stars,
    b.review_count,
    b.categories,
    b.is_open,
    b.hours_monday, b.hours_tuesday, b.hours_wednesday,
    b.hours_thursday, b.hours_friday, b.hours_saturday, b.hours_sunday,
    u.user_id,
    u.name          AS user_name,
    u.review_count  AS user_review_count,
    u.average_stars AS user_avg_stars
FROM reviews r
INNER JOIN businesses_filtered b ON r.business_id = b.business_id
LEFT  JOIN users u               ON r.user_id     = u.user_id;

SELECT COUNT(*) AS total_merged_rows FROM merged;

total_merged_rows
614283


## Cell 9 — Filter to Philadelphia Food Reviews

In [0]:
philly_df = spark.sql("""
    SELECT review_id, business_id, user_id, categories, city, review_text, review_stars
    FROM merged
    WHERE review_text IS NOT NULL
      AND LENGTH(review_text) > 50
      AND city = 'Philadelphia'
      AND (
          categories LIKE '%Restaurants%'
          OR categories LIKE '%Food%'
          OR categories LIKE '%Bars%'
          OR categories LIKE '%Cafes%'
          OR categories LIKE '%Nightlife%'
      )
""")

philly_df.createOrReplaceTempView("merged_philly")
print(f"Philadelphia food reviews ready: {philly_df.count():,}")

Philadelphia food reviews ready: 512,335


## Cell 10 — Install Dependencies

In [0]:
%pip install gensim nltk

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Cell 11 — Import Libraries

In [0]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag_sents
from gensim import corpora
from gensim.models.ldamodel import LdaModel

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

nltk_stopwords = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
print(f"Ready. NLTK stopwords: {len(nltk_stopwords)}")

[nltk_data] Downloading package stopwords to /home/spark-
[nltk_data]     fc3991b0-794e-4b8c-a865-a1/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /home/spark-
[nltk_data]     fc3991b0-794e-4b8c-a865-a1/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/spark-fc3991b0-794e-4b8c-a865-a1/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/spark-fc3991b0-794e-4b8c-a865-a1/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Ready. NLTK stopwords: 198


## Cell 12 — Sample 300K Reviews for LDA

In [0]:
review_sample = spark.sql("""
    SELECT review_id, business_id, user_id, review_text
    FROM merged_philly
""")

# Fixed seed = same 300K reviews every run (reproducible)
review_sample = review_sample.sample(fraction=0.51, seed=42)
review_pd = review_sample.toPandas()
print(f"Sampled {len(review_pd):,} reviews for LDA training")

Sampled 261,571 reviews for LDA training


## Cell 13 — Preprocess Text

**KEY FIX — why cuisine words are removed:**

The structure doc says Layer 2 already hard-filters restaurants by cuisine using the `categories` 
field (e.g. typing "Italian" → only Italian restaurants stay in the pool). If LDA also learns 
cuisine topics like "Mexican & Indian" or "Ramen & Sushi", those topic dimensions carry zero 
new information — they duplicate what `categories` already tells the recommender.

By removing cuisine words before LDA trains, we force it to discover what remains: 
atmosphere (cozy, loud, intimate), occasion (brunch, date night, late night), 
social context (solo, group, family). These are the dimensions Layer 1 cosine similarity 
actually needs to distinguish two Italian restaurants from each other by *vibe*, not cuisine.


In [0]:
# ── Generic Yelp noise (unchanged from v1) ───────────────────
generic_stopwords = {
    'good', 'great', 'really', 'place', 'like', 'just', 'dont',
    'also', 'even', 'would', 'back', 'went', 'came', 'got',
    'food', 'restaurant', 'much', 'time', 'think', 'know',
    'want', 'people', 'make', 'pretty', 'never', 'still',
    'well', 'always', 'definitely', 'nice', 'best', 'amazing',
    'love', 'delicious', 'better', 'though', 'something',
    'didnt', 'wasnt', 'could', 'since', 'going', 'said',
    'take', 'took', 'give', 'made', 'first', 'last', 'another',
    'every', 'super', 'cant', 'youre', 'theres', 'around',
    'little', 'told', 'asked', 'nothing', 'thing', 'however',
    'probably', 'maybe', 'right', 'times', 'come', 'find',
    'work', 'everything', 'enjoyed', 'loved', 'perfect',
    'excellent', 'awesome', 'recommend', 'favorite',
    'review', 'need', 'look', 'feel', 'kind', 'actually',
    'trying', 'hard', 'doesnt', 'thats', 'mean', 'whole',
    'front', 'might', 'name',
}

# ── NEW: Cuisine / food-type nouns removed ────────────────────
# WHY: Layer 2 cuisine hard-filter already uses `categories` field.
# LDA learning cuisine = duplicate signal, wastes topic slots.
# Removing forces LDA to find atmosphere, occasion, emotion instead.
cuisine_stopwords = {
    # Cuisine nationalities
    'mexican', 'italian', 'japanese', 'chinese', 'korean', 'thai',
    'indian', 'vietnamese', 'french', 'spanish', 'greek', 'mediterranean',
    'american', 'latin', 'caribbean', 'ethiopian', 'peruvian', 'turkish',
    'lebanese', 'moroccan', 'german', 'polish', 'irish',
    # Dish / food-type nouns
    'sushi', 'ramen', 'pho', 'taco', 'tacos', 'burrito', 'tamale',
    'pizza', 'pasta', 'lasagna', 'risotto', 'tiramisu', 'gelato',
    'burger', 'burgers', 'bbq', 'barbeque', 'barbecue', 'ribs', 'brisket',
    'dumpling', 'dumplings', 'dimsum', 'noodle', 'noodles', 'wonton',
    'steak', 'steakhouse', 'seafood', 'lobster', 'crab', 'shrimp',
    'sashimi', 'tempura', 'teriyaki', 'udon', 'miso',
    'falafel', 'hummus', 'shawarma', 'kebab', 'gyro',
    'curry', 'tikka', 'naan', 'samosa', 'biryani',
    'croissant', 'crepe', 'baguette',
    'cheesesteak', 'hoagie', 'pretzel',
    'sandwich', 'sandwiches', 'wrap', 'panini', 'subcategory',
    'soup', 'stew', 'chowder', 'bisque',
    'salad', 'salads', 'bowl', 'bowls',
    'chicken', 'beef', 'pork', 'lamb', 'salmon', 'tuna', 'tofu',
    'vegan', 'vegetarian', 'gluten',
}

all_stopwords = nltk_stopwords | generic_stopwords | cuisine_stopwords
print(f"Stopwords: {len(nltk_stopwords)} NLTK + {len(generic_stopwords)} generic + {len(cuisine_stopwords)} cuisine removed = {len(all_stopwords)} total")


def get_wordnet_pos(tag):
    if tag.startswith('J'):   return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else:                     return wordnet.NOUN


def preprocess_batch(texts):
    cleaned = [re.sub(r'[^a-zA-Z\s]', '', t.lower()).split() for t in texts]
    all_pos_tags = pos_tag_sents(cleaned)
    results = []
    for pos_tags in all_pos_tags:
        tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
        tokens = [t for t in tokens if t not in all_stopwords and len(t) > 3]
        results.append(tokens)
    return results


batch_size = 1000
all_tokens = []
total = len(review_pd)

for i in range(0, total, batch_size):
    batch = review_pd['review_text'].iloc[i:i+batch_size].tolist()
    all_tokens.extend(preprocess_batch(batch))
    if i % 10000 == 0:
        print(f"  Preprocessing: {i:,}/{total:,} ({round(i/total*100)}%)")

review_pd['tokens'] = all_tokens
print(f"\nDone! Sample tokens (no cuisine words): {review_pd['tokens'].iloc[0][:10]}")

Stopwords: 198 NLTK + 92 generic + 99 cuisine removed = 388 total
  Preprocessing: 0/261,571 (0%)
  Preprocessing: 10,000/261,571 (4%)
  Preprocessing: 20,000/261,571 (8%)
  Preprocessing: 30,000/261,571 (11%)
  Preprocessing: 40,000/261,571 (15%)
  Preprocessing: 50,000/261,571 (19%)
  Preprocessing: 60,000/261,571 (23%)
  Preprocessing: 70,000/261,571 (27%)
  Preprocessing: 80,000/261,571 (31%)
  Preprocessing: 90,000/261,571 (34%)
  Preprocessing: 100,000/261,571 (38%)
  Preprocessing: 110,000/261,571 (42%)
  Preprocessing: 120,000/261,571 (46%)
  Preprocessing: 130,000/261,571 (50%)
  Preprocessing: 140,000/261,571 (54%)
  Preprocessing: 150,000/261,571 (57%)
  Preprocessing: 160,000/261,571 (61%)
  Preprocessing: 170,000/261,571 (65%)
  Preprocessing: 180,000/261,571 (69%)
  Preprocessing: 190,000/261,571 (73%)
  Preprocessing: 200,000/261,571 (76%)
  Preprocessing: 210,000/261,571 (80%)
  Preprocessing: 220,000/261,571 (84%)
  Preprocessing: 230,000/261,571 (88%)
  Preprocessing:

## Cell 14 — Build Dictionary & Corpus

In [0]:
dictionary = corpora.Dictionary(review_pd['tokens'])

# Drop words in <10 reviews (too rare) or >50% of reviews (too common)
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(tokens) for tokens in review_pd['tokens']]

print(f"Vocabulary size: {len(dictionary):,}")
print(f"Corpus size    : {len(corpus):,}")

Vocabulary size: 18,524
Corpus size    : 261,571


## Cell 15 — Train LDA Model

25 topics. With cuisine removed, these slots now fill with atmosphere/occasion/context themes 
instead of food-type duplicates. Fixed seed=42 for reproducibility.


In [0]:
NUM_TOPICS = 25

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=5,
    alpha='auto',
    per_word_topics=True
)

print("LDA training complete!")
print(f"Topics: {NUM_TOPICS}")

LDA training complete!
Topics: 25


## Cell 16 — View Raw Topics

Look for atmosphere/occasion signals. You should no longer see cuisine words.  
Expected patterns: cozy, intimate, quiet, loud, vibe, patio, outdoor, brunch,  
date, night, morning, solo, group, friends, couple, wait, staff, happy hour, etc.


In [0]:
print("RAW TOPIC WORDS — read before labelling in Cell 17")
print("=" * 65)
for topic_id in range(NUM_TOPICS):
    words = lda_model.show_topic(topic_id, topn=12)
    word_list = [f"{w}({round(p,3)})" for w, p in words]
    print(f"\nTopic {topic_id:02d}: {[w for w,_ in lda_model.show_topic(topic_id,10)]}")
print("=" * 65)
print("Now go to Cell 17 and assign a label to each topic ID above.")

RAW TOPIC WORDS — read before labelling in Cell 17

Topic 00: ['order', 'wait', 'service', 'tell', 'star', 'minute', 'walk', 'long', 'leave', 'sure']

Topic 01: ['philly', 'staff', 'friendly', 'spot', 'visit', 'amaze', 'ever', 'service', 'many', 'area']

Topic 02: ['donut', 'vedge', 'hype', 'beilers', 'pound', 'gotta', 'truck', 'fusion', 'mini', 'refreshing']

Topic 03: ['wall', 'sprout', 'fire', 'creme', 'fondue', 'wear', 'dive', 'winter', 'brussels', 'photo']

Topic 04: ['coffee', 'market', 'shop', 'terminal', 'bagel', 'read', 'cafe', 'latte', 'milk', 'joes']

Topic 05: ['location', 'stay', 'bathroom', 'room', 'hotel', 'building', 'mojitos', 'worker', 'airport', 'indoor']

Topic 06: ['price', 'worth', 'portion', 'quality', 'small', 'high', 'size', 'decent', 'cheap', 'city']

Topic 07: ['brunch', 'breakfast', 'toast', 'sunday', 'waffle', 'pancake', 'diner', 'morning', 'benedict', 'bacon']

Topic 08: ['inside', 'outside', 'street', 'seating', 'outdoor', 'park', 'parking', 'across', 'wi

## Cell 17 — Label Topics

1. No cuisine type — those words were removed, topics won't contain them
2. No negative labels — "Complaints", "Mediocre" must not appear in user taste profiles
3. Every label answers "I'm someone who enjoys ___"
4. These labels show in Streamlit taste profile card — must be human-readable


In [0]:
# ─────────────────────────────────────────────────────────────
# READ CELL 16 OUTPUT FIRST — then update these labels.
# Keys (0-24) must match the topic IDs printed in Cell 16.
# ─────────────────────────────────────────────────────────────

topic_labels = {
    0:  "Service & Wait Time",
    1:  "Philly Neighborhood Gems",
    2:  "Food Trucks & Trendy Spots",
    3:  "Unique & Quirky Dining",
    4:  "Cafe, Coffee & Reading Terminal",
    5:  "Venue & Location Experience",
    6:  "Value & Portion Size",
    7:  "Brunch & Breakfast",
    8:  "Outdoor & Street Seating",
    9:  "Bar & Nightlife Crowd",
    10: "Desserts & Bakery",
    11: "Craft Beer & Sports Bars",
    12: "Quick & Fresh Lunch",
    13: "Customer Service Quality",
    14: "Spicy & Asian Flavors",
    15: "Small Plates & Cocktails",
    16: "Chef Specials & Platters",
    17: "Bar Vibes & Live Music",
    18: "Dinner & Happy Hour",
    19: "Comfort Food & Sandwiches",
    20: "Fine Dining & Tasting Menu",
    21: "Markets & Grocery Shopping",
    22: "Cocktail Bars & Speakeasy",
    23: "Overall Food Quality",
    24: "Casual Payment & Ordering",
}

# Verify labels against top words
print("Topic labels — verify against your Cell 16 output:")
print("=" * 65)
for tid, label in topic_labels.items():
    top5 = [w for w, _ in lda_model.show_topic(tid, topn=5)]
    print(f"Topic {tid:02d}  [{label}]")
    print(f"          words: {top5}")

Topic labels — verify against your Cell 16 output:
Topic 00  [Service & Wait Time]
          words: ['order', 'wait', 'service', 'tell', 'star']
Topic 01  [Philly Neighborhood Gems]
          words: ['philly', 'staff', 'friendly', 'spot', 'visit']
Topic 02  [Food Trucks & Trendy Spots]
          words: ['donut', 'vedge', 'hype', 'beilers', 'pound']
Topic 03  [Unique & Quirky Dining]
          words: ['wall', 'sprout', 'fire', 'creme', 'fondue']
Topic 04  [Cafe, Coffee & Reading Terminal]
          words: ['coffee', 'market', 'shop', 'terminal', 'bagel']
Topic 05  [Venue & Location Experience]
          words: ['location', 'stay', 'bathroom', 'room', 'hotel']
Topic 06  [Value & Portion Size]
          words: ['price', 'worth', 'portion', 'quality', 'small']
Topic 07  [Brunch & Breakfast]
          words: ['brunch', 'breakfast', 'toast', 'sunday', 'waffle']
Topic 08  [Outdoor & Street Seating]
          words: ['inside', 'outside', 'street', 'seating', 'outdoor']
Topic 09  [Bar & Nightli

## Cell 18 — Assign Topic Probabilities to All Reviews

In [0]:
rows = []
total = len(review_pd)
batch_size = 5000

for i in range(0, total, batch_size):
    batch = review_pd.iloc[i:i+batch_size]
    for _, row in batch.iterrows():
        bow = dictionary.doc2bow(row['tokens'])
        topic_probs = lda_model.get_document_topics(bow)
        for topic_id, prob in topic_probs:
            rows.append({
                'review_id':   row['review_id'],
                'business_id': row['business_id'],
                'user_id':     row['user_id'],
                'topic_id':    topic_id,
                'topic_label': topic_labels[topic_id],
                'topic_prob':  round(float(prob), 4)
            })
    if i % 50000 == 0:
        print(f"  Progress: {i:,}/{total:,} ({round(i/total*100)}%)")

topic_df = pd.DataFrame(rows)
print(f"\nDone! Total rows: {len(topic_df):,}")
print(f"Avg topics per review: {round(len(topic_df)/total, 1)}")
topic_df.head(5)

  Progress: 0/261,571 (0%)
  Progress: 50,000/261,571 (19%)
  Progress: 100,000/261,571 (38%)
  Progress: 150,000/261,571 (57%)
  Progress: 200,000/261,571 (76%)
  Progress: 250,000/261,571 (96%)

Done! Total rows: 3,854,659
Avg topics per review: 14.7


,review_id,business_id,user_id,topic_id,topic_label,topic_prob
0,ePXwsVqfkVZbI1jB0urCuQ,TFnGJlA5l_HDdzGDkNTdTA,eTj0BVNVQD32vNeCzoz66w,0,Service & Wait Time,0.1972
1,ePXwsVqfkVZbI1jB0urCuQ,TFnGJlA5l_HDdzGDkNTdTA,eTj0BVNVQD32vNeCzoz66w,1,Philly Neighborhood Gems,0.1417
2,ePXwsVqfkVZbI1jB0urCuQ,TFnGJlA5l_HDdzGDkNTdTA,eTj0BVNVQD32vNeCzoz66w,5,Venue & Location Experience,0.0326
3,ePXwsVqfkVZbI1jB0urCuQ,TFnGJlA5l_HDdzGDkNTdTA,eTj0BVNVQD32vNeCzoz66w,6,Value & Portion Size,0.0319
4,ePXwsVqfkVZbI1jB0urCuQ,TFnGJlA5l_HDdzGDkNTdTA,eTj0BVNVQD32vNeCzoz66w,8,Outdoor & Street Seating,0.0110


## Cell 19 — Save to msbabigdata.default.lda_topic_results

In [0]:
result_spark = spark.createDataFrame(topic_df)

result_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("msbabigdata.default.lda_topic_results")

print("Saved to msbabigdata.default.lda_topic_results")

# Quick summary
spark.sql("""
    SELECT
        topic_label,
        COUNT(*)              AS review_count,
        ROUND(AVG(topic_prob), 4) AS avg_prob
    FROM msbabigdata.default.lda_topic_results
    GROUP BY topic_label
    ORDER BY review_count DESC
""").show(25, truncate=False)

Saved to msbabigdata.default.lda_topic_results
+-------------------------------+------------+--------+
|topic_label                    |review_count|avg_prob|
+-------------------------------+------------+--------+
|Dinner & Happy Hour            |261571      |0.0941  |
|Service & Wait Time            |261571      |0.2704  |
|Philly Neighborhood Gems       |261571      |0.1361  |
|Overall Food Quality           |261476      |0.0487  |
|Comfort Food & Sandwiches      |261470      |0.072   |
|Value & Portion Size           |261212      |0.0518  |
|Quick & Fresh Lunch            |260846      |0.0491  |
|Bar Vibes & Live Music         |254699      |0.0353  |
|Fine Dining & Tasting Menu     |253084      |0.0322  |
|Outdoor & Street Seating       |175319      |0.0224  |
|Desserts & Bakery              |158715      |0.0264  |
|Spicy & Asian Flavors          |141863      |0.0248  |
|Markets & Grocery Shopping     |137598      |0.021   |
|Craft Beer & Sports Bars       |134775      |0.0255  |
|

## Cell 20 — Save Enriched Business Metadata

Saves `is_open` and all 7 `hours_*` fields into a separate metadata table.  
feature engineering notebook joins from here to build `business_profiles`.


In [0]:
business_meta_spark = spark.sql("""
    SELECT
        business_id, name, address, city, state, postal_code,
        categories, stars, review_count, is_open,
        hours_monday, hours_tuesday, hours_wednesday,
        hours_thursday, hours_friday, hours_saturday, hours_sunday
    FROM businesses_filtered
""")

business_meta_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("msbabigdata.default.business_metadata")

count = business_meta_spark.count()
print(f"Saved msbabigdata.default.business_metadata — {count:,} open businesses")
print("Columns:", business_meta_spark.columns)

Saved msbabigdata.default.business_metadata — 2,800 open businesses
Columns: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'categories', 'stars', 'review_count', 'is_open', 'hours_monday', 'hours_tuesday', 'hours_wednesday', 'hours_thursday', 'hours_friday', 'hours_saturday', 'hours_sunday']


## Cell 21 — Final Check: Confirm Tables Written

In [0]:
lda_rows  = spark.table("msbabigdata.default.lda_topic_results").count()
meta_rows = spark.table("msbabigdata.default.business_metadata").count()

print("=" * 55)
print("LDA PIPELINE COMPLETE")
print("=" * 55)
print(f"  msbabigdata.default.lda_topic_results  : {lda_rows:,} rows")
print(f"  msbabigdata.default.business_metadata  : {meta_rows:,} open businesses")
print()
print("Saloni — update your Cell 3 to read from:")
print("  spark.table('msbabigdata.default.lda_topic_results')")
print("  spark.table('msbabigdata.default.business_metadata')")
print("=" * 55)

LDA PIPELINE COMPLETE
  msbabigdata.default.lda_topic_results  : 3,854,659 rows
  msbabigdata.default.business_metadata  : 2,800 open businesses

Saloni — update your Cell 3 to read from:
  spark.table('msbabigdata.default.lda_topic_results')
  spark.table('msbabigdata.default.business_metadata')
